In [25]:
import pandas as pd

train_df = pd.read_csv("/Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/Users/vivekkumarkamal/Projects/Coding/smart-mcq-solver-challenge/test.csv")

X = train_df.drop('answer', axis=1)
y = train_df['answer']

In [26]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [27]:
import string

df = pd.read_csv('train.csv')
translator = str.maketrans('', '', string.punctuation)

unique_words = set()
for text in df['prompt']:
    cleaned_text = str(text).lower().translate(translator)
    words = cleaned_text.split()
    unique_words.update(words)

print("Total unique words:", len(unique_words))


Total unique words: 859


In [28]:

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Row where id == 1
row_id_1 = df[df['id'] == 1].iloc[0]
prompt_id_1 = row_id_1['prompt']

translator = str.maketrans('', '', string.punctuation)

def process_prompt(prompt):
    cleaned = str(prompt).lower().translate(translator)
    words = cleaned.split()
    filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return words, filtered_words

words_id_1, filtered_id_1 = process_prompt(prompt_id_1)

print("For row with id == 1 (index 0):")
print("Original:", prompt_id_1)
print("Cleaned words:", words_id_1)
print("Filtered words:", filtered_id_1)
print("Count of filtered words:", len(filtered_id_1))



For row with id == 1 (index 0):
Original: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Cleaned words: ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
Filtered words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Count of filtered words: 13


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_texts = []
for _, row in df.iterrows():
    combined = " ".join([
        str(row['prompt']), 
        str(row['A']), 
        str(row['B']), 
        str(row['C']), 
        str(row['D']), 
        str(row['E'])
    ])
    combined_texts.append(combined)

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_texts)

print("Vocabulary size:", len(vectorizer.get_feature_names_out())) # Output: 2762


Vocabulary size: 2762


In [30]:
from sklearn.metrics.pairwise import cosine_similarity

row_1 = df[df['id'] == 1].iloc[0]
prompt = row_1['prompt']
option_a = row_1['A']

combined_texts_2000 = [
    " ".join([str(r['prompt']), str(r['A']), str(r['B']), str(r['C']), str(r['D']), str(r['E'])])
    for _, r in df.iterrows()
]
vectorizer_2000 = TfidfVectorizer(stop_words='english').fit(combined_texts_2000)

vec_prompt_2000 = vectorizer_2000.transform([prompt])
vec_a_2000 = vectorizer_2000.transform([option_a])
sim_2000 = cosine_similarity(vec_prompt_2000, vec_a_2000)[0][0]
print(f"Scenario A Similarity: {sim_2000:.4f}") # Output: 0.2720

Scenario A Similarity: 0.2720


In [31]:
combined_texts = [
    " ".join([str(r['prompt']), str(r['A']), str(r['B']), str(r['C']), str(r['D']), str(r['E'])])
    for _, r in df.iterrows()
]
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

options_cols = ['A', 'B', 'C', 'D', 'E']
correct_count = 0

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    similarities = []
    for opt in options_cols:
        opt_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
    
    predicted_letter = options_cols[np.argmax(similarities)]
    
    if predicted_letter == row['answer']:
        correct_count += 1

accuracy_percent = (correct_count / len(df)) * 100
print(f"Percentage of correct matches: {accuracy_percent:.2f}%") # Output: 13.55%


Percentage of correct matches: 13.55%


In [32]:
import numpy as np

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(1, k)

# inputs
ground_truth = 'C'
predictions = ['C', 'A', 'B']

#  AP@3
score = apk(ground_truth, predictions, k=3)
print(f"MAP33 Score: {score}")


MAP33 Score: 1.0


In [33]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(1, k)

# inputs
ground_truth = 'B'
predictions = ['D', 'B', 'E']

# AP@3
score = apk(ground_truth, predictions, k=3)
print(f"MAP@3 Score: {score}")


MAP@3 Score: 0.5


In [34]:
# frequencies of each correct answer
value_counts = df['answer'].value_counts()
print("Answer frequencies:")
print(value_counts)

top_3 = value_counts.index[:3].tolist()
print(f"\nTop 3 most frequent answers (Predictions): {top_3}")

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(1, k)

scores = [apk(actual, top_3, k=3) for actual in df['answer']]
mean_score = sum(scores) / len(scores)

print(f"\nOverall MAP@3 score: {mean_score}")

Answer frequencies:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Top 3 most frequent answers (Predictions): ['B', 'C', 'A']

Overall MAP@3 score: 0.42125


In [35]:

combined_texts = [
    " ".join([str(r['prompt']), str(r['A']), str(r['B']), str(r['C']), str(r['D']), str(r['E'])])
    for _, r in df.iterrows()
]
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

options_cols = ['A', 'B', 'C', 'D', 'E']

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(1, k)

scores = []
for idx, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    similarities = []
    for opt in options_cols:
        opt_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
        
    similarities = np.array(similarities)
    sorted_indices = np.argsort(-similarities, kind='stable')
    top_3_preds = [options_cols[i] for i in sorted_indices[:3]]
    
    scores.append(apk(row['answer'], top_3_preds, k=3))

final_map3 = sum(scores) / len(scores)
print(f"Final average MAP@3 score: {final_map3:.6f}")


Final average MAP@3 score: 0.295667
